# FlowerClf — b3 inference (notebook submission)

Loads the offline-trained `tf_efficientnet_b3_ns @224` checkpoint (attached dataset),
runs hflip-TTA inference over the competition **test** TFRecords, writes `submission.csv`.
Internet is OFF: timm is pinned from bundled wheels (deps already in the Kaggle image).

In [ ]:
import subprocess, sys
WHEELS = '/kaggle/input/flowerclf-b3-weights/wheels'
try:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-index',
                    '--find-links', WHEELS, 'timm==1.0.27'], check=True)
except Exception as e:
    print('offline timm install skipped:', e)
import timm; print('timm', timm.__version__)

In [ ]:
import io, glob
import numpy as np, pandas as pd, torch
import tensorflow as tf
from PIL import Image
from timm.data import resolve_data_config, create_transform

COMP = '/kaggle/input/tpu-getting-started'
WEIGHTS = '/kaggle/input/flowerclf-b3-weights/tf_efficientnet_b3_ns_r224.pt'
RES = 224
BATCH = 64
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device', device)

ckpt = torch.load(WEIGHTS, map_location='cpu')
model = timm.create_model(ckpt['model'], pretrained=False, num_classes=104)
model.load_state_dict(ckpt['state_dict'])
model.eval().to(device)
cfg = resolve_data_config({}, model=model)
eval_tf = create_transform(input_size=(3, RES, RES), is_training=False,
                           mean=cfg['mean'], std=cfg['std'],
                           interpolation=cfg.get('interpolation', 'bicubic'),
                           crop_pct=cfg.get('crop_pct', 0.95))

In [ ]:
files = sorted(glob.glob(f'{COMP}/tfrecords-jpeg-{RES}x{RES}/test/*.tfrec'))
assert files, 'no test tfrecords found'
spec = {'image': tf.io.FixedLenFeature([], tf.string),
        'id': tf.io.FixedLenFeature([], tf.string)}
ds = tf.data.TFRecordDataset(files, num_parallel_reads=tf.data.AUTOTUNE)

ids, prob_chunks, imgs, batch_ids = [], [], [], []

@torch.no_grad()
def flush():
    if not imgs:
        return
    x = torch.stack(imgs).to(device)
    with torch.autocast('cuda', enabled=device == 'cuda'):
        logits = model(x) + model(torch.flip(x, dims=[3]))  # hflip TTA
    prob_chunks.append(torch.softmax(logits.float(), 1).cpu().numpy())
    ids.extend(batch_ids)
    imgs.clear(); batch_ids.clear()

for raw in ds:
    ex = tf.io.parse_single_example(raw, spec)
    img = Image.open(io.BytesIO(ex['image'].numpy())).convert('RGB')
    imgs.append(eval_tf(img))
    batch_ids.append(ex['id'].numpy().decode('utf-8'))
    if len(imgs) >= BATCH:
        flush()
flush()
probs = np.concatenate(prob_chunks)
print('test probs', probs.shape)

In [ ]:
sample = pd.read_csv(f'{COMP}/sample_submission.csv')
sub = pd.DataFrame({'id': ids, 'label': probs.argmax(1).astype(int)})
assert set(sub['id']) == set(sample['id'].astype(str)), 'id mismatch vs sample_submission'
sub = sub.set_index('id').loc[sample['id'].astype(str)].reset_index()
sub.to_csv('submission.csv', index=False)
print('wrote submission.csv', sub.shape, 'classes used:', sub['label'].nunique())
sub.head()